# Notebook C – Sensitivity check: LightGBM without `required_car_parking_spaces`

None of the 7,306 deduplicated bookings with a parking request was cancelled, so the field may be recorded at check-in. This notebook reruns LightGBM under the same 5-fold protocol as Notebook A (same folds, clipping, scaling and SMOTE inside the training fold), with the configuration chosen most often in Notebook A, once with and once without the parking feature. It runs on a CPU in a few minutes.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, lightgbm as lgb, time
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
SEED=42
df=pd.read_csv('hotel_bookings.csv').drop_duplicates().reset_index(drop=True)
df=df.drop(columns=['reservation_status','reservation_status_date','assigned_room_type','booking_changes','agent','company'])
df['children']=df['children'].fillna(0); df['country']=df['country'].fillna('Unknown')
cc=df['country'].value_counts(); df.loc[df['country'].isin(cc[cc<100].index),'country']='Other'
df['total_guests']=df.adults+df.children+df.babies; df['total_nights']=df.stays_in_weekend_nights+df.stays_in_week_nights
df['was_waitlisted']=(df.days_in_waiting_list>0).astype(int); df['is_family']=((df.children+df.babies)>0).astype(int)
df=df[df.total_guests>0].reset_index(drop=True)
cat=['hotel','arrival_date_month','meal','country','market_segment','distribution_channel','reserved_room_type','deposit_type','customer_type']
y=df.is_canceled.values; X0=pd.get_dummies(df.drop(columns=['is_canceled']),columns=cat,dtype=float).astype('float32')
cfg=dict(num_leaves=127,learning_rate=0.03,min_child_samples=10,colsample_bytree=0.6,subsample=1.0,reg_lambda=0.0)
for drop in [False,True]:
    X=X0.drop(columns=['required_car_parking_spaces']) if drop else X0
    num=[c for c in X.columns if not any(c.startswith(k+'_') for k in cat)]
    res=[]
    for tr,te in StratifiedKFold(5,shuffle=True,random_state=SEED).split(X,y):
        Xtr,Xte=X.iloc[tr].copy(),X.iloc[te].copy(); up=Xtr[num].quantile(.99)
        Xtr[num]=Xtr[num].clip(upper=up,axis=1); Xte[num]=Xte[num].clip(upper=up,axis=1)
        Xi,Xv,yi,yv=train_test_split(Xtr,y[tr],test_size=.1,stratify=y[tr],random_state=SEED)
        sc=MinMaxScaler(); Xi[num]=sc.fit_transform(Xi[num]); Xv[num]=sc.transform(Xv[num]); Xte[num]=sc.transform(Xte[num])
        Xs,ys=SMOTE(random_state=SEED).fit_resample(Xi.values,yi)
        m=lgb.LGBMClassifier(n_estimators=2000,subsample_freq=1,random_state=SEED,n_jobs=-1,verbose=-1,**cfg)
        m.fit(Xs,ys,eval_set=[(Xv.values,yv)],eval_metric='binary_logloss',callbacks=[lgb.early_stopping(50,verbose=False)])
        p=m.predict_proba(Xte.values)[:,1]; res.append([accuracy_score(y[te],p>=.5),f1_score(y[te],p>=.5),roc_auc_score(y[te],p)])
    r=np.array(res); print('drop parking' if drop else 'with parking', r.mean(0).round(4), r.std(0).round(4))


with parking [0.843  0.7017 0.9061] [0.0024 0.0042 0.0029]


drop parking [0.834  0.6794 0.8927] [0.0023 0.0044 0.0029]


Each printed line gives accuracy, F1 and ROC-AUC: first the mean over the five folds, then the standard deviation. Removing the parking feature lowers ROC-AUC from 0.906 to 0.893 and F1 from 0.702 to 0.679 (Section 4.6 of the paper).